# Exercise 1

In [15]:
import random
from datetime import datetime, timedelta
from py_ecc.secp256k1 import secp256k1


# ============================================================
# TARGET PUBLIC KEY GIVEN IN THE EXERCISE
# ============================================================

target_pk = (
    109406275515507847258077495698865554479234815675118879924707993547930533500352,
    40753420948420416814450900739928784816122565151273253896471439870288961392299
)


# ============================================================
# SEARCH PERIOD
# The example/output in the exercise is dated 20/09/2026.
# We search the week 21/09/2026 -> 1/10 /2026.
# ============================================================

start = datetime(2026, 9, 20, 0, 0, 0)
end = datetime(2026, 10, 1, 0, 0, 0)

start_timestamp = int(start.timestamp())
end_timestamp = int(end.timestamp())


print("Starting search...")
print("Start:", start)
print("End  :", end)
print("Number of timestamps:", end_timestamp - start_timestamp)
print()

# ============================================================
# BRUTE FORCE THE POSSIBLE TIMESTAMPS
# ============================================================

for timestamp in range(start_timestamp, end_timestamp):

    # Reproduce the seed used by the original program
    random.seed(timestamp)

    # Reproduce the private key
    candidate_sk = random.getrandbits(250)

    # Reproduce the public key
    candidate_pk = secp256k1.multiply(
        secp256k1.G,
        candidate_sk
    )

    # Compare with the public key from the exercise
    if candidate_pk == target_pk:
        print("SK Found")
        print("Timestamp :", timestamp)
        print("Date:",datetime.fromtimestamp(timestamp))
        print("SK:", candidate_sk)
        print("PK:", candidate_pk)
        print("Target PK :", target_pk)
        break

else:
    print("No matching key found.")

Starting search...
Start: 2026-09-20 00:00:00
End  : 2026-10-01 00:00:00
Number of timestamps: 950400

SK Found
Timestamp : 1790064086
Date: 2026-09-22 05:01:26
SK: 956253356851452492442368958386777854833970623075634634788050358078859320426
PK: (109406275515507847258077495698865554479234815675118879924707993547930533500352, 40753420948420416814450900739928784816122565151273253896471439870288961392299)
Target PK : (109406275515507847258077495698865554479234815675118879924707993547930533500352, 40753420948420416814450900739928784816122565151273253896471439870288961392299)


# Exercise 2

# Exercise 3

## 3.1 $ \alpha $ revalado

we know that $r = \alpha - c sk \mod n $. from first we isolate $c*sk$ like this : 
$$
c*sk = \alpha - r \mod n
$$

and then we devide by $c$ but since division doesn't truly exist its a multiplication by $c^{-1}$ and that gives us : 
$$
sk = (\alpha - r)*c^{-1}
$$


In [17]:
from py_ecc.secp256k1 import secp256k1

r=15215428533696616148478039431373436389500692627721265210089498777217667288867
c=77010669529568219686736112771359268396136143145059193730683690451398633016132
alpha=1181627538534364262043658325835895088564903807539555334358958613613546510155

cinv=pow(c,-1,secp256k1.N)

sk=((alpha-r)*cinv) % secp256k1.N   
if r==(alpha-c*sk) % secp256k1.N and sk <secp256k1.N:
    print("sk trouvé:",sk)
    

sk trouvé: 1266447806139624086483338558152700256280527111037723544892245543766122361329


## 3.2 $ \alpha $ s proximos

In [21]:
from py_ecc.secp256k1 import secp256k1
import hashlib, random, calendar
from datetime import datetime

N, G = secp256k1.N, secp256k1.G

m1, m2 = "Mensagem 1", "Mensagem 2"
c1 = 17574914162398755519845112831929872204794539148722560696530206738132537352309
r1 = 63702942183361152324349542819753229638304952016842535130701509206274961205200
c2 = 114394360644184457979898691066605911217074940116193361021583715765885654775545
r2 = 9307637531528717922211193650645999199252047737686460977332255290062646194943

def calc_c(message, alpha):
    """Reproduit le calcul de c de la fonction assinatura"""
    alphaG = secp256k1.multiply(G, alpha)
    h = hashlib.sha256()
    h.update(message.encode("UTF-8"))
    h.update(str(alphaG).encode("UTF-8"))
    return int(h.hexdigest(), 16)

def alpha_from_seed(seed):
    random.seed(seed)
    return random.getrandbits(250)

t = datetime(2026, 9, 23, 23, 26, 33)
base_utc = calendar.timegm(t.timetuple())

offsets = [-3, 0, 1, 2, -4, -5, -2, 3, -1, 4, 5, -6, -7, -8, 8, 9, 10]

found = None
for off in offsets:
    start = base_utc - off * 3600
    for seed in range(start - 2, start + 300):
        a1 = alpha_from_seed(seed)
        if calc_c(m1, a1) == c1:
            found = (seed, a1)
            break
    if found:
        print(f"Seed 1 found : {found[0]} (fuseau UTC{off:+d})")
        break

if not found:
    print("nothing found: enlarge the search period" )
else:
    seed1, alpha1 = found

    sk = ((alpha1 - r1) * pow(c1, -1, N)) % N
    print("sk =", sk)

    alpha2 = (r2 + c2 * sk) % N
    print("alpha2 < 2^250 ?", alpha2 < 2**250)

    for seed in range(seed1, seed1 + 600):
        if alpha_from_seed(seed) == alpha2:
            print("Seed 2 =", seed, f"(gap of {seed - seed1} s with seed 1)")
            break

Seed 1 found : 1790216795 (fuseau UTC-3)
sk = 36198492134533099687192079161405630886387847077163093049507249510875585818564
alpha2 < 2^250 ? True
Seed 2 = 1790216797 (gap of 2 s with seed 1)
